In [13]:
%%capture
!pip install groq chromadb pdfplumber sentence-transformers

In [2]:
import os
from google.colab import userdata
from groq import Groq
from sentence_transformers import SentenceTransformer

# Initialize Groq client
client = Groq(api_key=userdata.get('GROQ_API_KEY'))

# Initialize embedding model for Chroma DB
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [14]:
import pdfplumber
import chromadb
import os

def get_pdf_text(path):
    text = ""
    if not os.path.exists(path):
        return ""
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            content = page.extract_text()
            if content:
                text += content + " "
    return text.strip()

def word_chunker(text, chunk_size=500):
    words = text.split()
    for i in range(0, len(words), chunk_size):
        yield " ".join(words[i:i + chunk_size])

# Ingest new document
pdf_path = '/content/Seerat e Mustafa_new.pdf'
full_text = get_pdf_text(pdf_path)
chunks = list(word_chunker(full_text))

# Setup Chroma
chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection(name="seerah_collection")
except:
    pass
collection = chroma_client.create_collection(name="seerah_collection")

# Ingest chunks
for i, chunk in enumerate(chunks):
    collection.add(
        ids=[f"id_{i}"],
        embeddings=[embed_model.encode(chunk).tolist()],
        documents=[chunk]
    )
print(f"Ingested {len(chunks)} chunks into Chroma DB.")

Ingested 571 chunks into Chroma DB.


In [15]:
query = "Describe the events of the Battle of Badr according to the provided text."
MODEL_NAME = "llama-3.1-8b-instant"

# Baseline
baseline_response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": query}]
)

# RAG
results = collection.query(query_embeddings=[embed_model.encode(query).tolist()], n_results=3)
context = "\n".join(results['documents'][0])

rag_response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": f"Answer based ONLY on this context:\n{context}"},
        {"role": "user", "content": query}
    ]
)

print("--- BASELINE ---\n", baseline_response.choices[0].message.content)
print("\n--- RAG ---\n", rag_response.choices[0].message.content)

--- BASELINE ---
 You haven't provided any text, could you please provide the text so I can accurately describe the events of the Battle of Badr? If not, I can give you information based on my general knowledge about the Battle of Badr.

The Battle of Badr was a pivotal battle between the early Muslims and the Quraysh tribe, led by Abu Sufyan. It took place in 624 CE, in the valley of Badr, near Medina.

Here's a general overview of the events based on historical records:

- The Prophet Muhammad (peace be upon him) sent a group of 300 Muslims, led by Abu Bakr (the first caliph of Islam), to intercept a Quraysh caravan that was traveling from Syria to Mecca. The caravan was heavily guarded and contained valuable goods.
- The Quraysh, aware of the Muslim plan, decided to attack the Muslims by force. Abu Sufyan, the leader of the Quraysh, assembled a large army of around 1,000 men, but many of them decided not to join the fight, leaving Abu Sufyan with around 950 men.
- When the Muslim an